In [ ]:
## Holy grail theory
## v3 - all juriks : which tools carry NON-REDUNDANT signal
##
## gain importance is biased by cardinality + collinearity (RSX/DMX/VEL families are
## mutually correlated) -> it splits credit arbitrarily and cannot answer "which tool
## is non-redundant". So ranking is done by:
##   1. permutation importance on TEST (unbiased by cardinality)
##   2. SHAP on a sample (fair credit among correlated features, + direction)
##   3. stability across walk-forward folds (keep only what's robust)
## gain is printed too, but only as a rough cross-check, NOT the answer.

In [ ]:
import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.metrics import mean_absolute_error, average_precision_score

In [ ]:
RES_FILE    = 'ha-3sec-full-train-v3.pqt'
N_PERM_ROWS = 1_500_000      # permutation/SHAP subsample (test is large; full is slow)
N_SHAP_ROWS = 200_000
N_FOLDS     = 8              # for cross-fold importance stability

df = pd.read_parquet(RES_FILE)
print(df.info())

EXCLUDE = ('remaining', 'timestamp', 'segmentId', 'date')
feats = [c for c in df.columns if c not in EXCLUDE]
# leak-guard: nothing segment-derived or target-derived may enter the matrix
assert not any(c in feats for c in ('segmentLength', 'G', 'segmentId', 'remaining')), feats
assert not any('segment' in c.lower() for c in feats), [c for c in feats if 'segment' in c.lower()]
print(f'\n{len(feats)} features. excluded: {sorted(set(df.columns) - set(feats))}')

params = dict(objective='regression_l1', num_leaves=63, learning_rate=0.05,
              feature_fraction=0.8, bagging_fraction=0.8, bagging_freq=1,
              min_data_in_leaf=200, num_threads=16, verbose=-1)

In [ ]:
%%time

# ------------------------------------------------------------------ single 80/20 split
days = np.sort(df['date'].unique())
cutoff = days[-200]
train = df[df['date'] < cutoff]
test  = df[df['date'] >= cutoff].copy()
print(f'\ntrain rows {len(train):,}  test rows {len(test):,}  '
      f'train days {train["date"].nunique()}  test days {test["date"].nunique()}')
y_tr, y_te = train['remaining'].values, test['remaining'].values

# baseline: g-conditioned mean (train-fit, test-lookup)
g_mean = train.groupby('g')['remaining'].mean()
base_pred = test['g'].map(g_mean).fillna(train['remaining'].mean()).values
mae_base = mean_absolute_error(y_te, base_pred)

# full model
model = lgb.train(params, lgb.Dataset(train[feats].values, label=y_tr, feature_name=feats),
                  num_boost_round=600)
pred_full = model.predict(test[feats].values)
mae_model = mean_absolute_error(y_te, pred_full)

print(f'\nMAE baseline (g only) : {mae_base:.4f}')
print(f'MAE model (all feats) : {mae_model:.4f}')
print(f'skill over baseline   : {1 - mae_model / mae_base:+.4%}')

In [ ]:
%%time

# ------------------------------------------------------------------ g vs geometry decomposition
feats_nog = [c for c in feats if c != 'g']
m_a = lgb.train(params, lgb.Dataset(train[feats_nog].values, label=y_tr, feature_name=feats_nog),
                num_boost_round=600)
mae_a = mean_absolute_error(y_te, m_a.predict(test[feats_nog].values))
m_b = lgb.train(params, lgb.Dataset(train[['g']].values, label=y_tr, feature_name=['g']),
                num_boost_round=600)
mae_b = mean_absolute_error(y_te, m_b.predict(test[['g']].values))
print(f'\ng alone (tree)      : {mae_b:.4f}  skill {1-mae_b/mae_base:+.2%}')
print(f'geometry no-g       : {mae_a:.4f}  skill {1-mae_a/mae_base:+.2%}')
print(f'geometry + g (full) : {mae_model:.4f}  skill {1-mae_model/mae_base:+.2%}')

In [ ]:
# ------------------------------------------------------------------ stratified error (full model)
test['abserr'] = np.abs(pred_full - y_te)
print('\nerror by remaining (near flip):')
print(test.groupby(test['remaining'].clip(upper=15))['abserr'].mean().to_string())
print('\nerror by g (near entry):')
print(test.groupby(test['g'].clip(upper=15))['abserr'].mean().to_string())

In [ ]:
%%time

# ------------------------------------------------------------------ k-sweep (FULL feats incl g)
print('\nclassification lift (remaining <= k), full features:')
for k in (2, 3, 4, 5):
    ytr_k = (train['remaining'] <= k).astype(int).values
    yte_k = (test['remaining'] <= k).astype(int).values
    mk = lgb.train({**params, 'objective': 'binary'},
                   lgb.Dataset(train[feats].values, label=ytr_k, feature_name=feats),
                   num_boost_round=600)
    p = mk.predict(test[feats].values)
    br = yte_k.mean(); ap = average_precision_score(yte_k, p)
    print(f'  k={k}: base_rate {br:.3f}  AP {ap:.3f}  lift {ap/br:.2f}x')

In [ ]:
%%time

# ================================================================== 1. PERMUTATION IMPORTANCE (test)
# shuffle each feature in TEST, measure MAE increase. Unbiased by cardinality/collinearity.
rng = np.random.default_rng(0)
idx = rng.choice(len(test), size=min(N_PERM_ROWS, len(test)), replace=False)
Xte = test[feats].values[idx]
yte_s = y_te[idx]
base_mae_perm = mean_absolute_error(yte_s, model.predict(Xte))
perm = {}
for j, f in enumerate(feats):
    saved = Xte[:, j].copy()
    Xte[:, j] = rng.permutation(saved)
    perm[f] = mean_absolute_error(yte_s, model.predict(Xte)) - base_mae_perm  # +ve = feature matters
    Xte[:, j] = saved
print('\n=== permutation importance on test (MAE increase when shuffled; higher = more important) ===')
for f, v in sorted(perm.items(), key=lambda x: -x[1]):
    print(f'  {f:16s} {v:+.5f}')

In [ ]:
%%time

# ================================================================== 2. SHAP (sample)
sidx = rng.choice(len(test), size=min(N_SHAP_ROWS, len(test)), replace=False)
shap = model.predict(test[feats].values[sidx], pred_contrib=True)  # [n, n_feat+1], last col = base
shap_imp = np.abs(shap[:, :-1]).mean(axis=0)                        # mean |contribution| per feature
print('\n=== SHAP mean |contribution| (fair credit among correlated features) ===')
for f, v in sorted(zip(feats, shap_imp), key=lambda x: -x[1]):
    print(f'  {f:16s} {v:.5f}')
# direction: sign of correlation between feature value and its shap contribution
print('\nSHAP direction (corr between feature and its contribution; +=higher feat -> more remaining):')
Xs = test[feats].values[sidx]
for j, f in sorted(enumerate(feats), key=lambda t: -shap_imp[t[0]])[:15]:
    c = np.corrcoef(Xs[:, j], shap[:, j])[0, 1]
    print(f'  {f:16s} {c:+.3f}')

In [ ]:
%%time

# ================================================================== 3. CROSS-FOLD STABILITY
# permutation importance recomputed per walk-forward fold; keep only features stable across folds.
print('\n=== cross-fold permutation-importance stability ===')
buckets = np.array_split(days, N_FOLDS)
fold_imp = {f: [] for f in feats}
for kf in range(1, N_FOLDS):
    trd = np.concatenate(buckets[:kf]); ted = buckets[kf]
    tr = df[df['date'].isin(set(trd))]; te = df[df['date'].isin(set(ted))]
    mk = lgb.train(params, lgb.Dataset(tr[feats].values, label=tr['remaining'].values, feature_name=feats),
                   num_boost_round=600)
    si = rng.choice(len(te), size=min(300_000, len(te)), replace=False)
    Xf = te[feats].values[si]; yf = te['remaining'].values[si]
    bm = mean_absolute_error(yf, mk.predict(Xf))
    for j, f in enumerate(feats):
        sv = Xf[:, j].copy(); Xf[:, j] = rng.permutation(sv)
        fold_imp[f].append(mean_absolute_error(yf, mk.predict(Xf)) - bm); Xf[:, j] = sv

stab = [(f, np.mean(v), np.std(v)) for f, v in fold_imp.items()]
print(f'{"feature":16s} {"mean_imp":>10s} {"std":>9s} {"mean/std":>9s}  (high mean + low std = robustly important)')
for f, mu, sd in sorted(stab, key=lambda x: -x[1]):
    ratio = mu / sd if sd > 1e-9 else np.inf
    print(f'  {f:16s} {mu:+.5f} {sd:.5f} {ratio:8.1f}')

print('\nread: rank by PERMUTATION importance (not gain). SHAP resolves which of the correlated')
print('RSX/VEL/CFB variants carries independent credit. Cross-fold std flags unstable (spurious) ones.')
print('question answered = which Jurik tools are non-redundant, NOT whether skill rose (it won\'t much).')